# focus1000 生图审阅（任务3）

材料：`data/focus1000/gen_prompts.jsonl`（V6.0 维度采样出题）+ `gen_results.jsonl`（qwen-image-3.0-pro 出图记录）+ `gen_imgs/`。
产出持续累积中（生图进程后台跑），重跑各 cell 即可刷新。

In [ ]:
import json, random, html
from pathlib import Path
from IPython.display import HTML
from collections import Counter

BASE = Path('data/focus1000')
prompts = {json.loads(l)['prompt_id']: json.loads(l) for l in (BASE/'gen_prompts.jsonl').read_text().splitlines() if l.strip()}
results = [json.loads(l) for l in (BASE/'gen_results.jsonl').read_text().splitlines() if l.strip()]
ok = [r for r in results if not r.get('error')]
fail = [r for r in results if r.get('error')]
print(f"prompts: {len(prompts)} | 出图成功: {len(ok)} | 失败: {len(fail)}")
print('画幅分布:', dict(Counter(r['ratio'] for r in ok)))
print('尺寸分布:', dict(Counter((r.get('width'), r.get('height')) for r in ok)))
lv = Counter(p.get('level') for p in prompts.values() if p.get('gen_prompt'))
print('难度分布:', dict(lv))
cb = Counter(p.get('combo_type') for p in prompts.values() if p.get('gen_prompt'))
print('组合类型 Top:', cb.most_common(8))

## 随机抽样审阅（改 seed 重跑换一批）

In [ ]:
SEED = 20260902
N = 12              # 每批看几张
SHOW_PROMPT = True  # False 只看图，快速过图

rng = random.Random(SEED)
batch = rng.sample(ok, min(N, len(ok)))
cards = []
for r in batch:
    p = prompts[r['prompt_id']]
    img = BASE / r['file']
    lines = []
    if SHOW_PROMPT:
        lines.append(f"<b>{html.escape(r['prompt_id'])}</b> | {r['ratio']} | {r.get('width')}x{r.get('height')} | {p.get('level')} | {p.get('combo_type')}")
        lines.append(f"scene={','.join(p.get('scene_types') or [])} premise={','.join(p.get('premise_types') or [])} hops={len(p.get('hop_types') or [])}")
        lines.append(f"<div style='color:#555'>{html.escape(p['gen_prompt'])}</div>")
        concl = p.get('key_visual_conclusions') or []
        if concl:
            lines.append("<div style='color:#830'>视觉结论: " + html.escape('；'.join(concl)) + '</div>')
    else:
        lines.append(f"<b>{html.escape(r['prompt_id'])}</b> | {r['ratio']}")
    card = ("<div style='border:1px solid #ddd;border-radius:6px;padding:8px;margin:6px 0;display:flex;gap:12px;align-items:flex-start;background:#fff'>"
            f"<img src='{img}' loading='lazy' style='max-height:340px;max-width:420px;object-fit:contain;background:#f6f6f6;border-radius:4px'>"
            "<div style='min-width:0;flex:1;font-size:12.5px;line-height:1.55'>" + '<br>'.join(lines) + "</div></div>")
    cards.append(card)
HTML(''.join(cards))

## 按指定实例审阅（改名字）

In [ ]:
INSTANCE = '软银 NAO'    # 改成想看的实例名

sel = [r for r in ok if r['instance'] == INSTANCE]
print(f'{INSTANCE}: {len(sel)} 张')
cards = []
for r in sel:
    p = prompts[r['prompt_id']]
    card = ("<div style='border:1px solid #ddd;border-radius:6px;padding:8px;margin:6px 0;display:flex;gap:12px;background:#fff'>"
            f"<img src='{BASE / r['file']}' loading='lazy' style='max-height:340px;max-width:420px;object-fit:contain;background:#f6f6f6;border-radius:4px'>"
            f"<div style='min-width:0;flex:1;font-size:12.5px'><b>{r['prompt_id']}</b> | {r['ratio']} | {p.get('level')} | {p.get('combo_type')}<br>"
            f"<div style='color:#555'>{html.escape(p['gen_prompt'])}</div></div></div>")
    cards.append(card)
if cards:
    display = HTML(''.join(cards))
else:
    display = f'{INSTANCE} 暂无图'
display

## 失败记录

In [ ]:
print(f'失败 {len(fail)} 条') if not fail else Counter(r['error'][:60] for r in fail).most_common(10)